# 02 — State Homeless Rate Map

Choropleth map showing overall homeless rate per 10,000 residents by state (HUD 2023 PIT count).

In [1]:
from pathlib import Path
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

ROOT = Path().resolve().parent
df = pd.read_csv(ROOT / 'data' / 'processed' / 'merged_state_data.csv')
print(f'Loaded {len(df)} states')

Loaded 51 states


In [2]:
fig = px.choropleth(
    df,
    locations='state_postal',
    locationmode='USA-states',
    color='homeless_rate_per_10k',
    scope='usa',
    hover_name='state_name',
    hover_data={'state_postal': False, 'homeless_rate_per_10k': ':.1f', 'total_homeless': ':,', 'unsheltered_pct': ':.1f'},
    color_continuous_scale='Reds',
    title='Homeless Rate per 10,000 Residents by State (HUD 2023 PIT Count)',
    labels={'homeless_rate_per_10k': 'Rate per 10k'},
)
fig.update_layout(geo_bgcolor='lightblue')
fig.show()

In [3]:
df['unsheltered_rate_per_10k'] = (df['unsheltered_homeless'] / (df['population_millions'] * 1e6) * 10000).round(2)

fig2 = px.choropleth(
    df,
    locations='state_postal',
    locationmode='USA-states',
    color='unsheltered_rate_per_10k',
    scope='usa',
    hover_name='state_name',
    hover_data={'state_postal': False, 'unsheltered_rate_per_10k': ':.2f', 'unsheltered_pct': ':.1f'},
    color_continuous_scale='OrRd',
    title='Unsheltered Homeless Rate per 10,000 Residents by State (HUD 2023)',
    labels={'unsheltered_rate_per_10k': 'Unsheltered per 10k'},
)
fig2.show()

In [4]:
top10 = df.nlargest(10, 'homeless_rate_per_10k')
bot10 = df.nsmallest(10, 'homeless_rate_per_10k')
combined = pd.concat([top10, bot10])

fig3 = px.bar(
    combined.sort_values('homeless_rate_per_10k', ascending=True),
    x='homeless_rate_per_10k',
    y='state_name',
    orientation='h',
    color='homeless_rate_per_10k',
    color_continuous_scale='Reds',
    title='Top 10 and Bottom 10 States: Homeless Rate per 10k (2023)',
    labels={'homeless_rate_per_10k': 'Rate per 10k', 'state_name': ''},
)
fig3.show()